In [5]:
import os
import tempfile
# conda activate DestVI
import anndata
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import os.path as osp 
# import destvi_utils
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
import scvi
import seaborn as sns
import torch
from scvi.model import CondSCVI, DestVI
import seaborn as sns
import torch

In [6]:
scvi.settings.seed = 0
sc.set_figure_params(figsize=(6, 6), frameon=False)
sns.set_theme()
torch.set_float32_matmul_precision("high")
save_dir = '/maiziezhou_lab2/yuling/label_Transfer/DestVI/Development'

data = sc.read_h5ad('/maiziezhou_lab2/yuling/Datasets/Development.h5ad')

st_data = data[data.obs['Batch'] == 'Stage44_telencephalon_rep2_FP200000239BL_E4',]
st_data.layers['counts'] = st_data.X
sc_adata = data[data.obs['Batch'] == 'Stage54_telencephalon_rep2_DP8400015649BRD6_2',]
sc_adata.layers['counts'] = sc_adata.X
 

Seed set to 0
/home/zhuy45/anaconda3/envs/DestVI/lib/python3.9/site-packages/anndata/_core/anndata.py:1818: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/tmp/ipykernel_1230809/1023829172.py:10: ImplicitModificationWarning: Setting element `.layers['counts']` of view, initializing view as actual.
  st_data.layers['counts'] = st_data.X
/tmp/ipykernel_1230809/1023829172.py:12: ImplicitModificationWarning: Setting element `.layers['counts']` of view, initializing view as actual.
  sc_adata.layers['counts'] = sc_adata.X


In [17]:
st_data.var_names

Index(['LOC101953204[nr]|ZNF850[hs] | AMEX60DD000003', 'AMEX60DD000005',
       'LOC108803380[nr]|ZNF268[hs] | AMEX60DD000007',
       'FZD10 | AMEX60DD000016', 'PARPI_0010196[nr] | AMEX60DD000029',
       'SLC15A4 | AMEX60DD000036', 'TMEM132C | AMEX60DD000038',
       'TMEM132B | AMEX60DD000044', 'AACS | AMEX60DD000048',
       'BRI3BP | AMEX60DD000050',
       ...
       'CD44 | AMEX60DDU001040132', 'AMEX60DDU001040377', 'AMEX60DDU001040439',
       'CMTM8 | AMEX60DDU001040738', 'AMEX60DDU001041052',
       'OJAV_G00236990[nr] | AMEX60DDU001041344', 'AMEX60DDU001041346',
       'AMEX60DDU001041390', 'MEA1 | AMEX60DDU001042129',
       'PRDM7[nr]|ZNF662[hs] | AMEX60DDU001042172'],
      dtype='object', name='new_name', length=12704)

In [13]:
sc_adata.write_h5ad('/maiziezhou_lab2/yuling/Datasets/development_sc.h5ad')

In [4]:

scvi.settings.seed = 0
 
CondSCVI.setup_anndata(sc_adata, layer = "counts", labels_key = "Annotation")
sc_model = CondSCVI(sc_adata, weight_obs=False)
sc_model.view_anndata_setup()
sc_model.train()
DestVI.setup_anndata(st_data, layer="counts")
st_model = DestVI.from_rna_model(st_data, sc_model)
st_model.view_anndata_setup()
st_model.train(max_epochs=2500)
st_data.obsm["proportions"] = st_model.get_proportions()
df = st_data.obsm["proportions"].copy()
df.to_csv(osp.join(save_dir,"proportions.csv"), index=True)  # keeps spot barcodes as the first column


Seed set to 0
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
/home/zhuy45/anaconda3/envs/DestVI/lib/python3.9/site-packages/scvi/data/fields/_base_field.py:64: UserWarning: adata.layers[counts] does not contain unnormalized count data. Are you sure this is what you want?
  self.validate_field(adata)


Anndata setup with scvi-tools version 1.1.6.post2.

Setup via `CondSCVI.setup_anndata` with arguments:

{'labels_key': 'Annotation', 'layer': 'counts'}

     Summary Statistics     
┏━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Summary Stat Key ┃ Value ┃
┡━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│     n_cells      │ 2929  │
│     n_labels     │  16   │
│      n_vars      │ 12704 │
└──────────────────┴───────┘

...

              Data Registry              
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Registry Key ┃  scvi-tools Location   ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│      X       │ adata.layers['counts'] │
│    ind_x     │ adata.obs['_indices']  │
└──────────────┴────────────────────────┘

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/zhuy45/anaconda3/envs/DestVI/lib/python3.9/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.


Epoch 2500/2500: 100%|██████████| 2500/2500 [08:19<00:00,  5.19it/s, v_num=1, train_loss_step=3.44e+6, train_loss_epoch=3.43e+6]

`Trainer.fit` stopped: `max_epochs=2500` reached.


Epoch 2500/2500: 100%|██████████| 2500/2500 [08:19<00:00,  5.01it/s, v_num=1, train_loss_step=3.44e+6, train_loss_epoch=3.43e+6]
